In [ ]:
from plot_lib import plot_data, plot_model, set_default

In [ ]:
set_default()

# Feature choosing

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn import svm
from xgboost import XGBRegressor,plot_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,log_loss
import matplotlib.pyplot as plt

In [ ]:
train = pd.read_csv('train_cleaning.csv')
label = pd.read_csv('label.csv')
test = pd.read_csv('test_cleaning.csv')

In [ ]:
train_str = train.select_dtypes(include=['object'])
train_int = train.select_dtypes(exclude=['object'])
test_str  = test.select_dtypes (include =['object'])
test_int  = test.select_dtypes (exclude =['object'])

In [ ]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()

In [ ]:
for i in range(train_str.shape[1]):
    le.fit(train_str.iloc[:,i])
    train_str.iloc[:,i] = le.transform(train_str.iloc[:,i])

In [ ]:
train = pd.concat([train_int,train_str],axis=1)
label = pd.read_csv('label.csv')

In [ ]:
for i in range(test_str.shape[1]):
    le.fit(test_str.iloc[:,i])
    test_str.iloc[:,i] = le.transform(test_str.iloc[:,i])

In [ ]:
test = pd.concat([test_int,test_str],axis=1)

In [ ]:
train.to_csv('train_set.csv',index=False)
test.to_csv('test_set.csv',index=False)

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn import svm
from xgboost import XGBRegressor,plot_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,log_loss
import matplotlib.pyplot as plt

In [ ]:
train = pd.read_csv('train_set.csv')
label = pd.read_csv('label.csv')
test = pd.read_csv('test_set.csv')

In [ ]:
train = train.drop('Id', axis=1)
test = test.drop('Id', axis=1)

## 1. Using Lasso to degenerate the parameters

In [ ]:
sgd_l1 = SGDRegressor(max_iter=30, penalty="l1", random_state=1,alpha=1e-2)
sgd_l1.fit(train, label)

In [ ]:
sgd_l1.coef_

In [ ]:
sgd_l1.coef_.min(), sgd_l1.coef_.max()

In [ ]:
good_feats = np.abs(np.array(sgd_l1.coef_)) > 1e+12 # lower bound alpha=1e-6

print("Features reduced from %10d to %10d" % (train.shape[1], int(good_feats.sum())))

In [ ]:
train_l1_select = train.loc[:, good_feats]
test_l1_select = test.loc[:, good_feats]

## 2. Using XGBooster to plot importance

In [ ]:
model = XGBRegressor(max_depth=5,n_estimators=1000,learning_rate=1e-2)
model.fit(train,label)

In [ ]:
print(model.feature_importances_)

In [ ]:
plot_importance(model,max_num_features=25)
plt.show()

In [ ]:
good_feats_xgb = np.abs(np.array(model.feature_importances_)) > 1e-2
print("Features reduced from %10d to %10d" % (train.shape[1], int(good_feats_xgb.sum())))

In [ ]:
train_xgb_select = train.loc[:, good_feats_xgb]
test_xgb_select = test.loc[:, good_feats_xgb]

In [ ]:
train_xgb_select.to_csv('train_29feautre.csv',index=False)
test_xgb_select.to_csv('test_29feautre.csv',index=False)

# Model training

### 1.lasso selection

In [ ]:
x_train,x_val,y_train,y_val = train_test_split(train_l1_select,label,test_size = 0.3,random_state = 2)

In [ ]:
model = XGBRegressor(max_depth=5,learning_rate=1e-2,n_estimators=1000)
model.fit(x_train,y_train,\
          eval_metric='rmse',\
          eval_set=[(x_val,y_val)],\
          early_stopping_rounds=200,\
          verbose=1)

In [ ]:
pred = model.predict(test_l1_select)
submission = pd.read_csv('sample_submission.csv')
submission.SalePrice = pred
submission.to_csv('submission_l1_select.csv',index=False)

### 2. xgb selection

In [ ]:
x_train,x_val,y_train,y_val = train_test_split(train_xgb_select,label,test_size = 0.3,random_state = 3)

In [ ]:
model = XGBRegressor(max_depth=5,learning_rate=1e-2,n_estimators=1000)
model.fit(x_train,y_train,\
          eval_metric='rmse',\
          eval_set=[(x_val,y_val)],\
          early_stopping_rounds=200,\
          verbose=2)

In [ ]:
pred = model.predict(test_xgb_select)
submission = pd.read_csv('sample_submission.csv')
submission.SalePrice = pred
submission.to_csv('submission_xgb_select.csv',index=False)

### Ensemble

In [ ]:
pred1 = pd.read_csv('submission_xgb_select.csv')
pred2 = pd.read_csv('submission_l1_select.csv')

In [ ]:
pred = 0.7*pred1.SalePrice.values + 0.3*pred2.SalePrice.values

In [ ]:
submission = pd.read_csv('sample_submission.csv')
submission.SalePrice = pred
submission.to_csv('submission_ensemble.csv',index=False)